# DeepGuard AI -- GPU Training on Colab (Disconnect-Proof)

**Resilient training pipeline** — saves to Google Drive after every epoch.
If disconnected, just re-run and it resumes from the last saved checkpoint.

- **Runtime**: Go to `Runtime > Change runtime type > T4 GPU`
- **Time**: ~25-35 minutes on T4 GPU
- **Output**: Trained `.h5` model auto-saved to Google Drive

---

## Step 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/DeepGuard_AI_Models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive save dir: {DRIVE_DIR}')

## Step 1: GPU Check + Clone

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {len(gpus)}')
for gpu in gpus:
    print(f'  {gpu}')
if not gpus:
    print('\n*** WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU ***')

In [ ]:
import os
if not os.path.exists('Deepfake-Detection-Project'):
    !git clone https://github.com/vishnuwadkar/Deepfake-Detection-Project.git
else:
    !cd Deepfake-Detection-Project && git pull origin main
%cd Deepfake-Detection-Project
!pip install -q mtcnn tqdm scikit-learn

## Step 2: Download Dataset

In [ ]:
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API configured!')

In [ ]:
import os, shutil
from pathlib import Path

dst_check = Path('data/processed/train/real')
if dst_check.exists() and len(list(dst_check.glob('*'))) > 1000:
    print(f'Dataset already exists ({len(list(dst_check.glob("*")))} images). Skipping download.')
else:
    !pip install -q kaggle
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data/raw --unzip
    src_root = Path('data/raw/real_vs_fake/real-vs-fake')
    dst_root = Path('data/processed')
    for split in ['train', 'valid', 'test']:
        for cls in ['real', 'fake']:
            src, dst = src_root/split/cls, dst_root/split/cls
            if not dst.exists() and src.exists():
                shutil.copytree(str(src), str(dst))
                print(f'  {split}/{cls}: {len(list(dst.glob("*"))):,}')
    print('\nDataset ready!')

## Step 3: DISCONNECT-PROOF Training

This cell trains the model and **saves to Google Drive after every epoch**.
If your runtime disconnects, just re-run from Step 0 — it will **resume from the last checkpoint**.

In [ ]:
import tensorflow as tf
import numpy as np
import os, shutil
from pathlib import Path
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.optimizers import Adam

import src.config as cfg
from src.model import build_model, CUSTOM_OBJECTS, unfreeze_top_layers

# === CONFIG (full dataset, optimized for T4 GPU) ===
cfg.MAX_TRAIN_SAMPLES = None    # ALL 100K images
cfg.BATCH_SIZE = 64
EPOCHS_P1 = 12
EPOCHS_P2 = 15
LR_P1 = 3e-4
LR_P2 = 1e-5

DRIVE_DIR = '/content/drive/MyDrive/DeepGuard_AI_Models'
DRIVE_CKPT = os.path.join(DRIVE_DIR, 'deepfake_detector.h5')
DRIVE_P1   = os.path.join(DRIVE_DIR, 'deepfake_detector_phase1.h5')
LOCAL_CKPT = 'models/deepfake_detector.h5'
os.makedirs('models', exist_ok=True)

# === DATASETS ===
print('Loading datasets...')
from src.train import make_datasets, get_class_weights
train_ds, val_ds, class_names = make_datasets()
class_weights = get_class_weights()
print(f'Classes: {class_names}')

# === COSINE DECAY ===
def cosine_lr(epoch, lr, total, base):
    return float(1e-7 + 0.5 * (base - 1e-7) * (1 + np.cos(np.pi * epoch / max(total, 1))))

# === CHECK FOR EXISTING CHECKPOINT (RESUME SUPPORT) ===
phase1_done = os.path.exists(DRIVE_P1)
has_checkpoint = os.path.exists(DRIVE_CKPT)

if phase1_done:
    print(f'\n*** Phase 1 checkpoint found on Drive! Skipping to Phase 2. ***')
    model = tf.keras.models.load_model(
        DRIVE_P1, custom_objects={**CUSTOM_OBJECTS, 'tf': tf}, compile=False
    )
    print(f'Loaded Phase 1 model: {model.count_params():,} params')
elif has_checkpoint:
    print(f'\n*** Checkpoint found on Drive! Resuming... ***')
    model = tf.keras.models.load_model(
        DRIVE_CKPT, custom_objects={**CUSTOM_OBJECTS, 'tf': tf}, compile=False
    )
    print(f'Loaded checkpoint: {model.count_params():,} params')
else:
    model = None

# ============================================================
# PHASE 1: Train head (skip if Phase 1 checkpoint exists)
# ============================================================
if not phase1_done:
    print(f'\n{"="*60}')
    print(f'  PHASE 1: Training classification head ({EPOCHS_P1} epochs)')
    print(f'{"="*60}')

    if model is None:
        model = build_model(trainable_base=False)
        model.summary(print_fn=lambda x: None)  # suppress
        print(f'Built fresh model: {model.count_params():,} params')

    model.compile(
        optimizer=Adam(learning_rate=LR_P1),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=cfg.LABEL_SMOOTHING),
        metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')],
    )

    # Save to BOTH local and Drive after every epoch
    class DriveSaver(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            self.model.save(DRIVE_CKPT)
            print(f'  >> Saved to Google Drive (epoch {epoch+1})')

    callbacks_p1 = [
        EarlyStopping(monitor='val_auc', patience=5, mode='max', restore_best_weights=True, verbose=1),
        ModelCheckpoint(LOCAL_CKPT, monitor='val_auc', save_best_only=True, mode='max', verbose=1),
        LearningRateScheduler(lambda e, lr: cosine_lr(e, lr, EPOCHS_P1, LR_P1)),
        DriveSaver(),
    ]

    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P1,
                   class_weight=class_weights, callbacks=callbacks_p1, verbose=1)

    best_auc = max(h1.history.get('val_auc', [0]))
    best_acc = max(h1.history.get('val_accuracy', [0]))
    print(f'\n  Phase 1 done! val_auc={best_auc:.4f}, val_acc={best_acc:.4f}')

    # Save Phase 1 backup to Drive
    model.save(DRIVE_P1)
    print(f'  Phase 1 backup saved to Drive')

# ============================================================
# PHASE 2: Fine-tune top 50 layers
# ============================================================
print(f'\n{"="*60}')
print(f'  PHASE 2: Fine-tuning top {cfg.FINETUNE_LAYERS} layers ({EPOCHS_P2} epochs)')
print(f'{"="*60}')

model = unfreeze_top_layers(model, n_layers=cfg.FINETUNE_LAYERS, new_lr=LR_P2)

class DriveSaver2(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        self.model.save(DRIVE_CKPT)
        print(f'  >> Saved to Google Drive (P2 epoch {epoch+1})')

callbacks_p2 = [
    EarlyStopping(monitor='val_auc', patience=7, mode='max', restore_best_weights=True, verbose=1),
    ModelCheckpoint(LOCAL_CKPT, monitor='val_auc', save_best_only=True, mode='max', verbose=1),
    LearningRateScheduler(lambda e, lr: cosine_lr(e, lr, EPOCHS_P2, LR_P2)),
    DriveSaver2(),
]

h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P2,
               class_weight=class_weights, callbacks=callbacks_p2, verbose=1)

best_auc = max(h2.history.get('val_auc', [0]))
best_acc = max(h2.history.get('val_accuracy', [0]))
print(f'\n  Phase 2 done! val_auc={best_auc:.4f}, val_acc={best_acc:.4f}')

# Final save
model.save(LOCAL_CKPT)
model.save(DRIVE_CKPT)
print(f'\n  FINAL MODEL SAVED to Drive and local!')
print(f'  Even if runtime dies now, your model is safe on Google Drive.')

## Step 4: Evaluate

In [ ]:
from src.evaluate import evaluate
evaluate()

from IPython.display import Image, display
if os.path.exists('models/evaluation_report.png'):
    display(Image('models/evaluation_report.png', width=900))
    shutil.copy2('models/evaluation_report.png', DRIVE_DIR)
    print('Report saved to Drive')

## Step 5: Convert to TF.js + Download

In [ ]:
!pip install -q tensorflowjs
!python scripts/convert_model.py

import zipfile
from google.colab import files

model_dir = Path('extension/model')
if model_dir.exists() and any(model_dir.iterdir()):
    with zipfile.ZipFile('tfjs_model.zip', 'w') as z:
        for f in model_dir.iterdir():
            z.write(f, f.name)
    shutil.copy2('tfjs_model.zip', DRIVE_DIR)
    print('TF.js model saved to Drive!')

# Download everything
for f in [LOCAL_CKPT, 'models/evaluation_report.png', 'tfjs_model.zip']:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024*1024)
        print(f'Downloading: {os.path.basename(f)} ({size:.1f} MB)')
        files.download(f)